# Stage 1b — What does each column mean, and which values does it hold?

**Purpose.** Before asking questions we need to know every column: what it means and which values it really contains. This notebook prints, for each of the three raw tables, the data type, the number of missing and distinct values, and example values. For columns with few distinct values it prints all of them with their counts.

**Where the meanings come from.** The author only defines some columns. Every meaning below carries a tag:

| Tag | Meaning |
|---|---|
| **[A]** | The author states it in the Kaggle description or column descriptions |
| **[I]** | Inferred from the column name or from sample values, **not confirmed**. Check it against the output below |

After running the notebook, change an **[I]** to **[A]** or **[V]** (verified by us) only when the printed values support it, and fix the meaning when they don't.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1 and 2 take seconds. Part 3 reads the 80M-row list table once and takes a few minutes; set `RUN_LISTS = False` in the setup cell to skip the full read.

## Data dictionary

### Anime table — `AnimeList.csv`, one row is an anime (31 columns)

| Column | Meaning | Tag |
|---|---|---|
| `anime_id` | MyAnimeList id of the anime, the key of this table | [A] |
| `title` | Title (romanized) | [A] |
| `title_english` | English title | [A] |
| `title_japanese` | Japanese title | [A] |
| `title_synonyms` | Other names of the anime | [A] |
| `image_url` | URL of the cover image | [A] |
| `type` | Format of the anime, for example TV or OVA | [I] |
| `source` | What the anime is based on, for example Manga or Original | [I] |
| `episodes` | Number of episodes (a 0 may mean unknown) | [I] |
| `status` | Airing status, for example Finished Airing | [I] |
| `airing` | Whether the anime is currently airing (True/False) | [I] |
| `aired_string` | Air dates as text, for example `Apr 6, 2018 to ?` | [I] |
| `aired` | Air dates in a second format, to be checked | [I] |
| `duration` | Length of one episode as text, for example `24 min. per ep.` | [I] |
| `rating` | Age or content rating | [I] |
| `score` | Average score given by MAL users | [I] |
| `scored_by` | Number of users who scored it | [I] |
| `rank` | Rank by score. Stored as a decimal number but holds integers only, because of missing values | [A] |
| `popularity` | Rank by popularity | [I] |
| `members` | Number of users who have it on their list | [I] |
| `favorites` | Number of users who marked it as favorite | [I] |
| `background` | Free-text background information | [I] |
| `premiered` | Season it premiered in, for example Spring 2012 | [I] |
| `broadcast` | Weekday and time of broadcast | [I] |
| `related` | Related anime (sequel, prequel, adaptation) stored as text of a dictionary | [I] |
| `producer` | Producers, several in one cell | [I] |
| `licensor` | Licensors, several in one cell | [I] |
| `studio` | Animation studio(s) | [A] |
| `genre` | Genres, several in one cell | [A] |
| `opening_theme` | Opening songs | [I] |
| `ending_theme` | Ending songs | [I] |

### User table — `UserList.csv`, one row is a user (17 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user's name, used as the user key | [A] |
| `user_id` | Numeric id of the user | [I] |
| `user_watching` | Number of anime on the user's list with status watching | [I] |
| `user_completed` | ... with status completed | [I] |
| `user_onhold` | ... with status on hold | [I] |
| `user_dropped` | ... with status dropped | [I] |
| `user_plantowatch` | ... with status plan to watch | [I] |
| `user_days_spent_watching` | Total days the user spent watching | [I] |
| `gender` | Gender | [A] |
| `location` | Location, free text typed by the user | [A] |
| `birth_date` | Birth date | [A] |
| `access_rank` | Unknown, appears empty | [I] |
| `join_date` | Registration date | [A] |
| `last_online` | Last date the user was online | [A] |
| `stats_mean_score` | The user's mean score | [I] |
| `stats_rewatched` | Number of rewatched anime | [I] |
| `stats_episodes` | Total episodes watched | [I] |

The five `user_*` count columns are described by the author only as "aggregated values from their anime lists".

### List table — `UserAnimeList.csv`, one row is a user–anime pair (11 columns)

| Column | Meaning | Tag |
|---|---|---|
| `username` | The user | [A] |
| `anime_id` | The anime | [A] |
| `my_watched_episodes` | Episodes the user has watched | [I] |
| `my_start_date` | Date the user started watching (`0000-00-00` may mean empty) | [I] |
| `my_finish_date` | Date the user finished | [I] |
| `my_score` | Score the user gave, 1–10. **0 means not rated**, since the 1–10 scale has no 0 | [A] |
| `my_status` | Status code: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch. **The file stores numbers, the words come from the description.** Other codes are unexplained | [A] |
| `my_rewatching` | Whether the user is rewatching | [I] |
| `my_rewatching_ep` | Episode of the rewatch | [I] |
| `my_last_updated` | Timestamp of the last update of this row | [A] |
| `my_tags` | Free-text tags written by the user | [I] |

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips the full read of the 80M-row list table
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))


def profile(df, n_examples=4):
    """One row per column: type, missing, distinct values and a few example values."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "distinct": int(s.nunique(dropna=True)),
            "examples": s.dropna().astype(str).str.slice(0, 45).unique()[:n_examples].tolist(),
        })
    return pd.DataFrame(rows).set_index("column")


def show_values(df, max_distinct=25):
    """Print every distinct value, with its count, for columns that have few distinct values."""
    for col in df.columns:
        distinct = df[col].nunique(dropna=False)
        if distinct <= max_distinct:
            counts = df[col].value_counts(dropna=False)
            try:
                counts = counts.sort_index()
            except TypeError:
                pass
            print(f"\n--- {col}: {distinct} distinct values")
            print(counts.to_string())

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 1. Anime table

For every column: its type, how many values are missing, how many distinct values it has, and a few examples. Read the `examples` column slowly and compare it with the dictionary above.

In [2]:
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
print("shape:", anime.shape, "| anime_id unique:", anime.anime_id.is_unique)
print(profile(anime).to_string())

shape: (14478, 31) | anime_id unique: True
                  dtype  missing  missing_pct  distinct                                                                                                                                                                                      examples
column                                                                                                                                                                                                                                               
anime_id          int64        0         0.00     14478                                                                                                                                                                      [11013, 2104, 5262, 721]
title            object        0         0.00     14477                                                                                                                          [Inu x Boku SS, Seto no Hanayome, Shugo Ch

In [3]:
# Every distinct value of the columns that have few of them
show_values(anime)


--- type: 7 distinct values
type
Movie      2624
Music       849
ONA        1144
OVA        3612
Special    1948
TV         4271
Unknown      30

--- source: 16 distinct values
source
4-koma manga      224
Book               93
Card game          59
Digital manga      10
Game              607
Light novel       563
Manga            3120
Music             314
Novel             359
Original         3368
Other             408
Picture book       96
Radio               9
Unknown          4210
Visual novel      888
Web manga         150

--- status: 3 distinct values
status
Currently Airing      351
Finished Airing     13791
Not yet aired         336

--- airing: 2 distinct values
airing
False    14127
True       351

--- rating: 7 distinct values
rating
G - All Ages                      4541
None                               544
PG - Children                     1279
PG-13 - Teens 13 or older         5020
R - 17+ (violence & profanity)     997
R+ - Mild Nudity                   878
Rx - He

### Observations for part 1

**TODO (fill in after the run):**
- For each **[I]** column of the anime dictionary: does the output confirm the meaning? Change the tag or correct the text.
- Columns with few distinct values: which values are there, and are any unexpected (typos, placeholders such as `Unknown`)?
- Columns with many distinct values: what format do the examples show (comma-separated lists, dictionaries as text, dates as text)?

## 2. User table

In [4]:
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("shape:", users.shape, "| username unique:", users.username.is_unique)
print(profile(users).to_string())

shape: (302675, 17) | username unique: True
                            dtype  missing  missing_pct  distinct                                                                              examples
column                                                                                                                                                 
username                   object        0         0.00    302675                                        [karthiga, RedvelvetDaisuki, Damonashu, bskai]
user_id                     int64        0         0.00    302673                                                     [2255153, 1897606, 37326, 228342]
user_watching               int64        0         0.00       580                                                                       [3, 61, 45, 25]
user_completed              int64        0         0.00      2208                                                                   [49, 396, 195, 414]
user_onhold                 int64        0  

In [5]:
show_values(users)

print("\n--- location: top 15 of", users["location"].nunique(), "distinct values")
print(users["location"].value_counts().head(15).to_string())


--- gender: 4 distinct values
gender
Female         81374
Male          135181
Non-Binary      1245
NaN            84875

--- access_rank: 1 distinct values
access_rank
NaN    302675

--- location: top 15 of 55283 distinct values
location
Poland         2425
Germany        1947
Canada         1821
California     1510
Brazil         1459
France         1135
Philippines    1077
Australia      1032
Sweden         1013
Finland         898
Portugal        896
England         874
Singapore       863
USA             709
Texas           694


### Observations for part 2

**TODO (fill in after the run):** the same questions as part 1 for the user table. Which gender values exist, how messy is `location`, what do `birth_date`, `join_date` and `last_online` look like, and is `access_rank` really empty?

## 3. List table

The file has about 80 million rows, so it is read in two ways:

1. **A sample of the first 1,000,000 rows, all columns as text.** This only shows the *formats* of the columns. The file is ordered by user, so the first million rows belong to a small group of users and must not be used for statistics.
2. **A full read of the columns that hold few distinct values or numbers** (one pass, a few minutes).

In [6]:
sample = pd.read_csv(paths["UserAnimeList.csv"], nrows=1_000_000, dtype=str, **na_kwargs)
print("sample shape:", sample.shape, "| columns:", list(sample.columns))
print(profile(sample).to_string())
print()
print(sample.head(5).T.to_string())
del sample

sample shape: (1000000, 11) | columns: ['username', 'anime_id', 'my_watched_episodes', 'my_start_date', 'my_finish_date', 'my_score', 'my_status', 'my_rewatching', 'my_rewatching_ep', 'my_last_updated', 'my_tags']
                      dtype  missing  missing_pct  distinct                                                                                       examples
column                                                                                                                                                    
username             object        0         0.00      3065                                                 [karthiga, RedvelvetDaisuki, Damonashu, bskai]
anime_id             object        0         0.00     14453                                                                              [21, 59, 74, 120]
my_watched_episodes  object        0         0.00       743                                                                             [586, 26, 161, 70]
my_start_da

In [7]:
if RUN_LISTS:
    list_cols = {
        "my_status": "int8", "my_score": "int8", "my_rewatching": "category",
        "my_watched_episodes": "float32", "my_rewatching_ep": "float32",
        "my_start_date": "category", "my_finish_date": "category",
    }
    lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(list_cols), dtype=list_cols, **na_kwargs)
    print(f"rows: {len(lists):,}")

    status_names = {1: "watching", 2: "completed", 3: "on hold", 4: "dropped", 6: "plan to watch"}
    status_table = lists.my_status.value_counts().sort_index().to_frame("rows")
    status_table["meaning"] = status_table.index.map(status_names).fillna("unknown")
    status_table["share_pct"] = (status_table.rows / len(lists) * 100).round(3)
    print("\n--- my_status")
    print(status_table.to_string())

    print("\n--- my_score (0 = not rated)")
    print(lists.my_score.value_counts().sort_index().to_string())

    print("\n--- my_rewatching")
    print(lists.my_rewatching.value_counts(dropna=False).to_string())

    print("\n--- my_watched_episodes and my_rewatching_ep")
    print(lists[["my_watched_episodes", "my_rewatching_ep"]]
          .describe(percentiles=[.5, .99, .999]).round(1).to_string())

    for col in ["my_start_date", "my_finish_date"]:
        counts = lists[col].value_counts(dropna=False)
        print(f"\n--- {col}: {lists[col].nunique():,} distinct values | top 5:")
        print(counts.head(5).to_string())
        print("share of '0000-00-00':", f"{(lists[col] == '0000-00-00').mean():.1%}")
else:
    print("Skipped (RUN_LISTS = False)")

rows: 80,076,112

--- my_status
               rows        meaning  share_pct
my_status                                    
0               527        unknown      0.001
1           4057788       watching      5.067
2          50445526      completed     62.997
3           2985804        on hold      3.729
4           3194353        dropped      3.989
5                 6        unknown      0.000
6          19392104  plan to watch     24.217
33                3        unknown      0.000
55                1        unknown      0.000

--- my_score (0 = not rated)
my_score
0     33717790
1       321270
2       374744
3       619587
4      1296354
5      2905183
6      5456518
7     10379184
8     11394082
9      7897871
10     5713529

--- my_rewatching
my_rewatching
0      63288415
NaN    16741847
1         45850

--- my_watched_episodes and my_rewatching_ep
       my_watched_episodes  my_rewatching_ep
count           80076112.0        80076112.0
mean                  14.0               

### Observations for part 3

**TODO (fill in after the run):** the formats of `my_last_updated` and `my_tags`, what `my_rewatching` holds, whether `0000-00-00` is common in the start and finish dates, and whether `my_watched_episodes` has impossible values. Then update the tags in the list-table dictionary.